## Étape 6 (Optionnelle) : Aller plus loin : trois leviers, trois problèmes

Votre système marche, vous le présentez en revue interne. Trois remontées concrètes vont vous faire pivoter successivement : qualité plafonnée, coût d'entraînement, latence d'inférence. Chaque levier ML répond à un problème précis ; à vous de chiffrer son apport.

### 6.1. Finetunning end-to-end
---
Diagnostic. Les embeddings figés de l'étape 4 sont génériques : l'encodeur n'a jamais vu Banking77. Le classifieur ne peut compenser qu'à la marge.
**Réponse.** Finetuner tous les poids de DistilBERT + une tête de classification sur la tâche.

---

**Objectif du code :** 

1. Finetuner tous les poids de DistilBERT
    * DistilBERT est un modèle de langage pré-entraîné. Jusqu'ici nous avons utilisé un *Frozen Encoder* (paramètres gelés)
    * "Finetuner tous les poids" => tous les paramètres (poids) du modèle DistilBERT seront ajustés pendant l'entraînement du jeu de données spécifique.

2. Tête de classification
    * Une tête de classification est une couche supplémentaire ajoutée au-dessus du modèle DistilBERT pour produire une prédiction
    * Cette couche est initialement aléatoire et sera entraînée en même temps que les poids de DistilBERT.

On améliore les performances du modèle (prédiction d'intention) au domaine particulier de la banque où le vocabulaire et les structures de phrases diffèrent du corpus d'entraînement original.



In [3]:
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset
import torch
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# =========================================================
# 1. Chargement du dataset Banking77
# =========================================================
dataset = load_dataset("mteb/banking77")

# =========================================================
# 2. Chargement du tokenizer de DistilBERT
# =========================================================
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

# =========================================================
# 3. Prétraitement des données (tokenisation)
# =========================================================
def tokenize_function(texts):
    return tokenizer(texts["text"], padding="max_length", truncation=True, max_length=48)   # tester avec max_length=64 pour réduire le temps de traitement

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# =========================================================
# 4. Charger le modèle DistilBERT avec une tête de classification
# =========================================================
# Le nombre de labels est 77 pour Banking77
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=77,
)


# =========================================================
# 5. Définition les métriques d'évaluation
# =========================================================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

# =========================================================
# 6. Configuration des arguments d'entraînement
# =========================================================
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",             # fréquence d'évaluation du modèle : fin de chaque epoch
    learning_rate=2e-5,                # taille du pas d'apprentissage : pas standard pour DistillBert d'après la littérature
    per_device_train_batch_size=4,     # Nombre de texts traités simultanément pendant le train : pour réduire le temps de traitement => 4
    per_device_eval_batch_size=4,      # Nombre de texts traités simultanément pendant l'évaluation : pour réduire le temps de traitement => 4
    num_train_epochs=3,                # Passe complète sur l'ensemble des données d'entrainement
    weight_decay=0.01,                 # Coefficient de regularisation appliqué pendant l'entrainement (0.01 est la valeur standard recommandée)
    save_strategy="epoch",             # Définit la stratégie de sauvegarde des checkpoints du modèle => à la fin de chaque Epoch
    load_best_model_at_end=True,
    dataloader_pin_memory=False       # utilise uniquement le CPU (pour éviter d'avoir un warning)
)

# =========================================================
# 7. Initialiser le Trainer
# =========================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# =========================================================
# 8. Lancer l'entraînement
# =========================================================
trainer.train()

# =========================================================
# 9. Évaluer le modèle
# =========================================================
results = trainer.evaluate()
print(f"Résultats après fine-tuning : {results}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.410555,0.847396,0.818921,0.799705
2,0.389371,0.398516,0.895969,0.894291
3,0.206681,0.357470,0.908322,0.907938


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.206681,0.357470,3,0.908322,0.907938


Résultats après fine-tuning : {'eval_loss': 0.3574696481227875, 'eval_accuracy': 0.9083224967490247, 'eval_f1': 0.9079377465826622}


### 6.1. LoRA / PEFT
---

LoRA est une méthode de *parameter-efficient fine-tuning* : au lieu d’entraîner tous les poids du modèle, on ajoute de petites matrices entraînables à certains blocs, ce qui réduit le nombre de paramètres entraînables, accélère le fine-tuning et réduit l’usage mémoire. L’usage recommandé par Hugging Face consiste à créer un LoraConfig puis à envelopper le modèle avec get_peft_model().([huggingface.co](https://huggingface.co/docs/peft/developer_guides/lora), [github.com](https://github.com/huggingface/peft))

L’idée est donc de procéder selon les étapes suivantes :
1. geler l’encodeur DistilBERT (pas de mise à jour des poids de base) ;
2. ajouter des adaptateurs LoRA dans les couches d’attention ;
3. laisser entraînables LoRA + tête de classification.

C’est à priori le compromis recherché quand on veut réduire le coût d’entraînement tout en gardant une bonne performancedu modèle.

Installation de la librairie PEFT : `pip install peft`

In [10]:
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, f1_score

import numpy as np
import torch
import pandas as pd
import os
from datetime import datetime


# =========================================================
# 1. Chargement du dataset
# =========================================================
dataset = load_dataset("mteb/banking77")

# label_names = dataset["train"].features["label"].names
# num_labels = len(label_names)

# =========================================================
# 2. Tokenizer
# =========================================================
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

def tokenize_function(texts):
    return tokenizer(
        texts["text"],
        truncation=True,
        max_length=96,   # plus léger que 128 pour mon CPU / RAM - 48 - 96
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Important : colonnes utiles pour Trainer
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================================
# 3. Modèle de base
# =========================================================
# Charger le modèle DistilBERT avec une tête de classification
# Le nombre de labels est 77 pour Banking77
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=77,
)

# =========================================================
# 4. Geler l'encodeur DistilBERT
# =========================================================
for param in model.distilbert.parameters():
    param.requires_grad = False

# On garde la tête de classification entraînable
for param in model.pre_classifier.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

# =========================================================
# 5. Configuration LoRA
# =========================================================
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                       # rang faible = plus léger 8 - 16
    lora_alpha=32,              # 16 - 32
    lora_dropout=0.05,
    target_modules=["q_lin", "v_lin"],  # point de départ recommandé pour DistilBERT
    bias="none",
    modules_to_save=["pre_classifier", "classifier"],  # conserve la tête entraînable/sauvée
)

# Injection LoRA
model = get_peft_model(model, lora_config)

# Debug utile : combien de paramètres restent entraînables ?
model.print_trainable_parameters()

# =========================================================
# 6. Métriques
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_weighted = f1_score(labels, preds, average="weighted")
    f1_macro = f1_score(labels, preds, average="macro")

    return {
        "accuracy": acc,
        "f1_weighted": f1_weighted,
        "f1_macro": f1_macro,
    }

# =========================================================
# 7. Training arguments
# =========================================================
training_args = TrainingArguments(
    output_dir="./results_lora_distilbert",
    learning_rate=2e-4,                 # souvent plus haut qu’un full finetuning classique
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",              
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    dataloader_num_workers=2,
    dataloader_pin_memory=False,
    report_to="none",
)

# =========================================================
# 8. Trainer
# =========================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    #tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


# =========================================================
# 9. Log des entrainements (paramètres et métriques)
# =========================================================

def log_results(results, training_args, lora_config, tokenizer_config, training_time_sec, file_path="log_resuts.xlsx"):

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Construction d'une ligne de résultats
    row = {
        "timestamp": now,
        "model_name": model_name,

        # tokenizer
        "max_length": tokenizer_config.get("max_length"),

        # LoRA
        "lora_r": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
        "lora_dropout": lora_config.lora_dropout,
        "lora_targets": ",".join(lora_config.target_modules),

        # training
        "learning_rate": training_args.learning_rate,
        "batch_train": training_args.per_device_train_batch_size,
        "batch_eval": training_args.per_device_eval_batch_size,
        "epochs": training_args.num_train_epochs,

        # temps
        "training_time_sec": round(training_time_sec, 2),
        
        # metrics
        "eval_loss": results.get("eval_loss"),
        "eval_accuracy": results.get("eval_accuracy"),
        "eval_f1": results.get("eval_f1"),
    }

    df_new = pd.DataFrame([row])

    # Si fichier existe → append
    if os.path.exists(file_path):
        df_existing = pd.read_excel(file_path)
        df_final = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_final = df_new

    # Sauvegarde
    df_final.to_excel(file_path, index=False)

    print(f"✅ Entrainement enregistré dans {file_path}")


# =========================================================
# 10. Train
# =========================================================

start_time = datetime.now()

trainer.train()

end_time = datetime.now()
training_time_sec = (end_time - start_time).total_seconds()


# =========================================================
# 11. Eval
# =========================================================
results = trainer.evaluate()
print("Résultats :", results)


# =========================================================
# 12. Sauvegarde et log des résultats
# =========================================================
trainer.save_model("./distilbert_banking77_lora")
tokenizer.save_pretrained("./distilbert_banking77_lora")

log_results(
    results=results,
    training_args=training_args,
    lora_config=lora_config,
    tokenizer_config={"max_length": 48},  # ou ta variable
    training_time_sec=training_time_sec
)







Map:   0%|          | 0/9993 [00:00<?, ? examples/s]

Map:   0%|          | 0/3076 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 944,717 || all params: 67,957,402 || trainable%: 1.3902


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro
1,0.595878,0.495363,0.860208,0.856085,0.856094
2,0.459877,0.378849,0.899870,0.899428,0.899399
3,0.284820,0.361810,0.905397,0.905310,0.905260


Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted,F1 Macro
0.284820,0.361810,3,0.905397,0.905310,0.905260


Résultats : {'eval_loss': 0.3618101477622986, 'eval_accuracy': 0.9053966189856957, 'eval_f1_weighted': 0.9053102033321233, 'eval_f1_macro': 0.905260016920869}


PermissionError: [Errno 13] Permission denied: 'log_resuts.xlsx'